# Winsorização (Winsorization)

**Winsorização** é a técnica de *aparar* valores extremos substituindo-os por **limites de percentil** pré-definidos, em vez de remover linhas.  
Ajuda a reduzir o impacto de *outliers* em métricas (média, desvio-padrão, KPIs), preservando o tamanho da amostra.

---

## Ideia em 1 minuto

- Escolha percentis, por exemplo **P1** e **P99**.  
- Calcule limites: **L = P1**, **U = P99**.  
- Para cada valor `x`:
  - se `x < L` → vira `L`  
  - se `x > U` → vira `U`  
  - senão permanece `x`

Equivalente a:  
\[
x_{\text{wins}}=\min\left(\max(x, L), U\right)
\]

---

## Quando usar / evitar

**Use quando:**
- Existem erros grosseiros (ex.: preço 10×, `0` indevido) que distorcem estatísticas.
- Você quer **manter N** (não remover linhas) e estabilizar KPIs.

**Evite quando:**
- Valores extremos têm **significado de negócio** (p. ex., pedidos VIP altíssimos que você deseja medir).
- Você precisa estudar as caudas separadamente (considere *trimming* ou análises por segmento).

---

## Passo a passo (genérico)

1. Escolha percentis (comuns: **P01/P99** ou **P05/P95**).  
2. Calcule `L` e `U` no subconjunto relevante (ex.: por produto/canal).  
3. Substitua (*clamp*) os valores fora do intervalo por `L`/`U`.  
4. Documente limites e impacto (antes/depois).

---

## Exemplo — SQL (DuckDB)

```sql
-- 1) Limites globais P01/P99 de unit_price
WITH p AS (
  SELECT
    quantile_cont(unit_price, 0.01) AS p01,
    quantile_cont(unit_price, 0.99) AS p99
  FROM stg.sales
  WHERE unit_price IS NOT NULL
)
-- 2) Aplicar winsorização (clamp)
SELECT
  order_id,
  date_parsed,
  GREATEST(LEAST(unit_price, p.p99), p.p01) AS unit_price_wins,
  quantity,
  customer_id,
  channel_norm
FROM stg.sales, p;
```

**Por grupo (recomendado quando há heterogeneidade, ex.: por produto):**
```sql
WITH bounds AS (
  SELECT
    product_id,
    quantile_cont(unit_price, 0.01) AS p01,
    quantile_cont(unit_price, 0.99) AS p99
  FROM stg.sales
  WHERE unit_price IS NOT NULL
  GROUP BY product_id
)
SELECT
  s.order_id,
  s.date_parsed,
  GREATEST(LEAST(s.unit_price, b.p99), b.p01) AS unit_price_wins,
  s.product_id,
  s.quantity,
  s.customer_id,
  s.channel_norm
FROM stg.sales s
LEFT JOIN bounds b USING (product_id);
```

---

## Exemplo — Python (pandas)

```python
import pandas as pd

# Supõe df com coluna 'unit_price'
p01 = df['unit_price'].quantile(0.01)
p99 = df['unit_price'].quantile(0.99)

df['unit_price_wins'] = df['unit_price'].clip(lower=p01, upper=p99)
```

**Por grupo (produto):**
```python
def winsorize_series(s, a=0.01, b=0.99):
    lo, hi = s.quantile(a), s.quantile(b)
    return s.clip(lower=lo, upper=hi)

df['unit_price_wins'] = (
    df.groupby('product_id', group_keys=False)['unit_price']
      .apply(winsorize_series)
)
```

---

## Boas práticas

- **Combine com regras determinísticas** (ex.: `unit_price <= 0` → tratar antes de winsorizar).
- **Escolha o agrupamento certo** (produto, canal, loja) para limites mais realistas.
- **Logue limites e impacto** (ex.: percentis antes/depois, nº de linhas alteradas).
- **Documente percentis usados** e o racional (P01/P99 vs. P05/P95).

---

## Winsorização × Trimming

- **Winsorização:** **substitui** caudas por limites → **mantém N**.  
- **Trimming:** **remove** as caudas → **reduz N**.

> Em relatórios executivos, winsorizar costuma ser preferido para estabilizar KPIs sem perder observações.